# Renewables in Electricity Markets: System Perspective

This is the solution to Assignment 1 proposed in the course "Renewables in Electricity Markets" from DTU in 2025.

The chosen system is the IEEE 24-bus test reliability system, shown in the figure below. Data is in the file IEEE 24-bus reliability test system - data.xlsx. 

# ![System](img/System.png)

## Step 5: Balancing Market

In this step we go back a to a copper plate model in a single hour to simulate what would happen in the balancing market. The chosen hour will be 8 AM. 

### Input data

In [1]:
# Import the libraries to be used
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import linopy
np.random.seed(42)

#### Read data

In [2]:
# system data
data = pd.read_excel('data/IEEE 24-bus reliability test system - data.xlsx', sheet_name=None)
print(data.keys())

dict_keys(['Generators', 'Costs', 'SystemLoadProfile', 'LoadDist', 'BidTypes', 'Lines', 'WindFarms', 'LineModification'])


In [3]:
#dict_keys(['Generators', 'Costs', 'SystemLoadProfile', 'LoadDist', 'BidTypes', 'Lines', 'WindFarms', 'LineModification'])
generators = data['Generators']
costs = data['Costs']
load_profile = data['SystemLoadProfile']
load_dist = data['LoadDist']
load_bid_types = data['BidTypes']
lines = data['Lines']
wind_farms = data['WindFarms']
line_modification = data['LineModification']

In [4]:
# windfarm data
wind_production_dk = pd.read_csv('data/ninja_wind_55.4104_12.3039_corrected.csv', skiprows=3)

#### Handle demand data

In [5]:
# Assign demand to each bus based on the load distribution
demand = load_profile.merge(load_dist, how='cross')
demand['Demand_MW'] = demand['System_demand_MW'] * demand['Percent_of_system_load'] / 100
demand['Hour'] = demand['Hour'] -1  # Adjust hour to be 0-indexed
demand.drop(columns=['System_demand_MW', 'Percent_of_system_load'], inplace=True)

In [6]:
# create demand bids: Load (id), Node (id), Quantity (MW), Price ($/MWh)
demand_bid = demand.merge(load_bid_types, left_on='Load_type', right_on='Load_type', how='left')
demand_bid['Quantity_MW'] = demand_bid['Demand_MW']*demand_bid['Quantity_perc']/100
demand_bid = demand_bid[['Bid', 'Load', 'Node', 'Hour', 'Load_type', 'Quantity_MW', 'Price_$/MWh']]
demand_bid['Load'] = 'L' + demand_bid['Load'].astype(str) 

#### Handle WindFarm data

In [7]:
#select 6 random days to simulate the wind production of each generator
np.random.seed(42)
wind_production_dk['local_time'] = pd.to_datetime(wind_production_dk['local_time'])
wind_production_dk['day'] = wind_production_dk['local_time'].dt.date
random_days = np.random.choice(wind_production_dk['day'].unique(), size=len(wind_farms), replace=False)
wind_production_sample = wind_production_dk[wind_production_dk['day'].isin(random_days)]
# assign one day to one zone windfarm
wind_production_sample['WindFarm'] = wind_production_sample['day'].apply(lambda x: random_days.tolist().index(x)+1)
wind_production_sample['Hour'] = wind_production_sample['local_time'].dt.hour

In [8]:
wind_forecast_generation = wind_production_sample.merge(wind_farms, on='WindFarm', how='left')
wind_forecast_generation['Generation_MW'] = wind_forecast_generation['Installed_Capacity_MW'] * wind_forecast_generation['electricity']
wind_forecast_generation = wind_forecast_generation[['WindFarm', 'Node', 'Hour', 'Generation_MW']].sort_values(['WindFarm', 'Hour'])
wind_forecast_generation['WindFarm'] = 'W' + wind_forecast_generation['WindFarm'].astype(str)
wind_forecast_generation.rename(columns={'Generation_MW': 'PForecast_MW', 'WindFarm': 'Unit'}, inplace=True)

#### Handle supply data

In [9]:
gen_constraints = generators.merge(costs, on='Unit')
gen_constraints['Unit'] = 'G' + gen_constraints['Unit'].astype(str)

### Modelling

#### Model 1. Market clearing for day ahead market

This model only solves the day ahead for 8 am.

In [10]:
# Sets
generators_set = pd.Index(gen_constraints['Unit'].unique(), name='Unit')
wind_farms_set = pd.Index(wind_forecast_generation['Unit'].unique(), name='Unit')
loads_set = pd.Index(demand_bid['Load'].unique(), name='Load')
bids_set = pd.Index(demand_bid['Bid'].unique(), name='Bid')

In [11]:
# Parameters
gen_constraints_xr = gen_constraints.set_index('Unit').to_xarray()
demand_bid_xr = demand_bid.set_index(['Load','Bid','Hour']).to_xarray()
wind_forecast_xr = wind_forecast_generation.set_index(['Unit','Hour']).to_xarray()

# select only hour 8
demand_bid_xr = demand_bid_xr.sel(Hour=8) 
wind_forecast_xr = wind_forecast_xr.sel(Hour=8) 


pmax = gen_constraints_xr['Pmax_MW'] # Maximum power output of each generator in MW
ci = gen_constraints_xr['Ci_$/MWh'] # Variable cost of each generator in $/MWh

wind_forecast = wind_forecast_xr['PForecast_MW'] # Forecasted power output of each wind farm in MW

demand_quantity = demand_bid_xr['Quantity_MW'].reindex(
    Load=loads_set, 
    Bid=bids_set, 
).fillna(0)

demand_price = demand_bid_xr['Price_$/MWh'].reindex(
    Load=loads_set, 
    Bid=bids_set,
).fillna(0)

In [12]:
model = linopy.Model()

# Decision variables: generation
g = model.add_variables(
    coords=[generators_set],
    name="dispatch"
)

# Decision variables: wind generation
w = model.add_variables(
    lower=0,
    coords=[wind_farms_set],
    name="wind_generation"
)

# Decision variables: demand
d = model.add_variables(
    lower=0,
    coords=[loads_set, bids_set],
    name="demand"
)

# Generators capacity constraints
capacity_constraints = model.add_constraints(
    g <= pmax,
    name="capacity_limit"
)

min_generation_constraint = model.add_constraints(
    g >= 0,
    name="min_capacity_limit"
)

# Wind generation constraints
wind_constraints = model.add_constraints(
    w<= wind_forecast,
    name="wind_dispatch"
)

# Demand constraints
demand_constraints = model.add_constraints(
    d <= demand_quantity,
    name="demand_limit"
)

# Power balance constraint
balance = model.add_constraints(
    g.sum(dim="Unit") + w.sum(dim="Unit") == d.sum(dim=["Load","Bid"]),
    name="power_balance"
)

# Then use these in your objective
model.add_objective(
    (g * ci).sum() - (d * demand_price).sum(),
    sense="min"
)

model.solve(solver_name="highs")

Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-zpy3zcu7 has 99 rows; 86 cols; 184 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [5e+00, 2e+03]
  Bound   [0e+00, 0e+00]
  RHS     [3e+00, 6e+02]
Presolving model
1 rows, 63 cols, 63 nonzeros  0s
1 rows, 14 cols, 14 nonzeros  0s
Dependent equations search running on 1 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
1 rows, 14 cols, 14 nonzeros  0s
Presolve reductions: rows 1(-98); columns 14(-72); nonzeros 14(-170) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
          1    -4.3396463771e+06 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-zpy3zcu7
Model status        : Optimal
Simplex   iterations: 1


('ok', 'optimal')

#### Model 2. Balancing market - need for upregulation

We use the solutions of the day-ahead market to set planned generation, but then we simulate that G10 has a forced outage and cannot generate, half of the wind farms (1 to 3) produce 15% less than forecasted, while the other half (4 to 6) produce 10% more than forecasted. We then solve the balancing market to see how the system reacts to these changes.

The remaining generators offer upward regulation at a price of day-ahead +10% of their variable cost, and downward regulation at a price of day-ahead -15% of their variable cost. The demand is inelastic in the balancing market.



In [13]:
p_gen = g.solution.to_dataframe()
p_dem = d.solution.to_dataframe()
p_wf = w.solution.to_dataframe()

p_gen_bm = p_gen.copy()
p_dem_bm = p_dem.copy()
p_wf_bm = p_wf.copy()

outage_gen = 'G10'

p_gen_bm.loc[outage_gen] = 0 # G10 has a forced outage and cannot generate
p_wf_bm.iloc[:3] = p_wf_bm.iloc[:3]*0.85
p_wf_bm.iloc[3:] = p_wf_bm.iloc[3:]*1.1

I'm not convinced by the price being the DA price +/- a percentage of the variable cost. It would be the variable cost of the generator plus a percentage for upward regulation and minus a percentage for downward regulation.

In [14]:
increase_price = ci*1.1
curtailment_cost = 500
down_price = ci*0.9

In [15]:
# Balancing model
def balancing_market_model(p_gen_bm, p_dem_bm, p_wf_bm, increase_price, curtailment_cost, down_price, outage_gen):
                           

    imbalance = p_dem_bm.sum() - p_gen_bm.sum() - p_wf_bm.sum()

    bm_model = linopy.Model()

    # Decision variables: generation
    g_up = bm_model.add_variables(
        coords=[generators_set],
        name="upregulation"
    )

    g_down = bm_model.add_variables(
        coords=[generators_set],
        name="downregulation"
    )

    # Decision variables: demand
    d_curt = bm_model.add_variables(
        lower=0,
        coords=[loads_set, bids_set],
        name="demand_curtailed"
    )

    # Generators capacity constraints
    up_regulation_capacity = bm_model.add_constraints(
        g_up <= (pmax - p_gen_bm.to_xarray())['solution'],
        name="capacity_limit"
    )

    min_up_regulation_capacity = bm_model.add_constraints(
        g_up >= 0,
        name="min_capacity_limit"
    )

    down_regulation_capacity = bm_model.add_constraints(
        g_down <= p_gen_bm.to_xarray()['solution'],
        name="down_capacity_limit"
    )

    min_down_regulation_capacity = bm_model.add_constraints(
        g_down >= 0,
        name="min_down_capacity_limit"
    )

    #set outage generation to 0
    gout_constraint_up = bm_model.add_constraints(
        g_up.sel(Unit=outage_gen) == 0,
        name=f"{outage_gen}_outage_up"
    )

    gout_constraint_down = bm_model.add_constraints(
        g_down.sel(Unit=outage_gen) == 0,
        name=f"{outage_gen}_outage_down"
    )

    # Demand constraints
    demand_constraints = bm_model.add_constraints(
        d_curt <= demand_quantity,
        name="demand_limit"
    )

    # Power balance constraint
    balance = bm_model.add_constraints(
        g_up.sum(dim="Unit") + d_curt.sum(dim=["Load","Bid"]) - g_down.sum(dim="Unit") == imbalance.solution,
        name="power_balance"
    )

    # Then use these in your objective
    bm_model.add_objective(
        (g_up * increase_price).sum() + (d_curt * curtailment_cost).sum() - (g_down * down_price).sum(),
        sense="min"
    )

    bm_model.solve(solver_name="highs")

    return bm_model

In [16]:
bm_up = balancing_market_model(p_gen_bm, p_dem_bm, p_wf_bm, increase_price, curtailment_cost, down_price, outage_gen)

Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-6hajt6kv has 119 rows; 92 cols; 210 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [5e+00, 5e+02]
  Bound   [0e+00, 0e+00]
  RHS     [3e+00, 6e+02]
Presolving model
1 rows, 57 cols, 57 nonzeros  0s
1 rows, 10 cols, 10 nonzeros  0s
Dependent equations search running on 1 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
1 rows, 10 cols, 10 nonzeros  0s
Presolve reductions: rows 1(-118); columns 10(-82); nonzeros 10(-200) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
          1     4.6124669250e+03 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-6hajt6kv
Model status        : Optimal
Simplex   iterations: 

#### Model 3. Balancing market - need for downregulation

In this case wind farms 1 to 3 are generating 15% less of what was expected and wind farms 4 to 6 are generating 20% more.

In [17]:
p_gen_bm_down = p_gen.copy()
p_dem_bm_down = p_dem.copy()
p_wf_bm_down = p_wf.copy()

outage_gen = 'G4'

p_gen_bm_down.loc[outage_gen] = 0 # G4 has a forced outage and cannot generate
p_wf_bm_down.iloc[:3] = p_wf_bm_down.iloc[:3]*0.85
p_wf_bm_down.iloc[3:] = p_wf_bm_down.iloc[3:]*1.2

In [18]:
bm_down = balancing_market_model(p_gen_bm_down, p_dem_bm_down, p_wf_bm_down, increase_price, curtailment_cost, down_price, outage_gen)

Running HiGHS 1.13.1 (git hash: 1d267d9): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-jw7imbjx has 119 rows; 92 cols; 210 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [5e+00, 5e+02]
  Bound   [0e+00, 0e+00]
  RHS     [3e+00, 6e+02]
Presolving model
1 rows, 57 cols, 57 nonzeros  0s
1 rows, 10 cols, 10 nonzeros  0s
Dependent equations search running on 1 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
1 rows, 10 cols, 10 nonzeros  0s
Presolve reductions: rows 1(-118); columns 10(-82); nonzeros 10(-200) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
          1    -7.3997550000e+01 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-jw7imbjx
Model status        : Optimal
Simplex   iterations: 

### Results and Analysis

These analysis will be focused on the profits for generators, given their day ahead positions and the balancing market outcomes.

#### General outcomes

In [19]:
dap = model.constraints['power_balance'].dual.values
s1_imbalance = (p_dem_bm.sum() - p_gen_bm.sum() - p_wf_bm.sum()).values[0]
s1_imbalance_price = bm_up.constraints['power_balance'].dual.values
s2_imbalance = (p_dem_bm_down.sum() - p_gen_bm_down.sum() - p_wf_bm_down.sum()).values[0]
s2_imbalance_price = bm_down.constraints['power_balance'].dual.values

print(f"Day ahead price: {dap:.2f} $/MWh")
print(f"\nScenario 1\n\tTotal imbalance: {s1_imbalance:.2f} MW")
print(f"\tImbalance price: {s1_imbalance_price:.2f} $/MWh")
print(f"\tUpregulation cost: {bm_up.objective.value:.2f} $")
print(f"\nScenario 2\n\tTotal imbalance: {s2_imbalance:.2f} MW")
print(f"\tImbalance price: {s2_imbalance_price:.2f} $/MWh")
print(f"\tDownregulation cost: {bm_down.objective.value:.2f} $")

Day ahead price: 10.89 $/MWh

Scenario 1
	Total imbalance: 314.09 MW
	Imbalance price: 22.77 $/MWh
	Upregulation cost: 4612.47 $

Scenario 2
	Total imbalance: -7.55 MW
	Imbalance price: 9.80 $/MWh
	Downregulation cost: -74.00 $


#### Profits for generators under one pricing scheme

In [20]:
# Analysis per generator per scenario -> Unit, cost, Scheduled generation, actual generation, DAM profit, BM profit, total profit, % change in profit compared to DAM

base_gen_df = pd.DataFrame({
    'Unit': generators_set,
    'Cost_$_per_MWh': ci.values,
    'Scheduled_Generation_MW': p_gen['solution'].values
})

# Add wind farm units
base_wind_df = pd.DataFrame({
    'Unit': wind_farms_set,
    'Cost_$_per_MWh': 0,
    'Scheduled_Generation_MW': p_wf['solution'].values
})

base_df = pd.concat([base_gen_df, base_wind_df], ignore_index=True)
base_df['DAM_Payments_$'] = base_df['Scheduled_Generation_MW'] * dap
base_df['DAM_Profit_$'] = base_df['Scheduled_Generation_MW'] * (dap - base_df['Cost_$_per_MWh'])

In [21]:
# # Scenario analysis
def compute_profits(base_df, p_gen_bm, p_wf_bm, imbalance_price, bm_model):
    gen_df = base_df.copy()
    actual_gen = np.concatenate([
        p_gen_bm['solution'].values + bm_model.variables['upregulation'].solution.values - bm_model.variables['downregulation'].solution.values,
        p_wf_bm['solution'].values
    ])
    gen_df['Actual_Generation_MW'] = actual_gen
    gen_df['Deviation_MW'] = gen_df['Actual_Generation_MW'] - gen_df['Scheduled_Generation_MW']
    gen_df['BM_Payments_$'] = gen_df['Deviation_MW'] * imbalance_price
    gen_df['BM_Profit_$'] = gen_df['Deviation_MW'] * (imbalance_price - gen_df['Cost_$_per_MWh'])
    gen_df['Total_Profit_$'] = gen_df['DAM_Profit_$'] + gen_df['BM_Profit_$']
    gen_df['Profit_Change_$'] = gen_df['Total_Profit_$'] - gen_df['DAM_Profit_$']
    gen_df['Profit_Change_%'] = (gen_df['Profit_Change_$'] / gen_df['DAM_Profit_$'].replace(0, np.nan)) * 100
    gen_df['Price_per_MWh'] = (gen_df['DAM_Payments_$'] + gen_df['BM_Payments_$'])/gen_df['Actual_Generation_MW']
    return gen_df

gen_s1 = compute_profits(base_df, p_gen_bm, p_wf_bm, s1_imbalance_price, bm_up)
gen_s2 = compute_profits(base_df, p_gen_bm_down, p_wf_bm_down, s2_imbalance_price, bm_down)

In [22]:
gen_s1.round(2)

,Unit,Cost_$_per_MWh,Scheduled_Generation_MW,DAM_Payments_$,DAM_Profit_$,Actual_Generation_MW,Deviation_MW,BM_Payments_$,BM_Profit_$,Total_Profit_$,Profit_Change_$,Profit_Change_%,Price_per_MWh
0,G1,13.32,-0.00,-0.00,0.00,152.00,152.00,3461.04,1436.40,1436.40,1436.40,NaN,22.77
1,G2,13.32,-0.00,-0.00,0.00,152.00,152.00,3461.04,1436.40,1436.40,1436.40,NaN,22.77
2,G3,20.70,-0.00,-0.00,0.00,3.46,3.46,78.90,7.17,7.17,7.17,NaN,22.77
3,G4,20.93,-0.00,-0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,NaN,NaN
4,G5,26.11,-0.00,-0.00,0.00,0.00,0.00,0.00,-0.00,0.00,0.00,NaN,NaN
5,G6,10.52,155.00,1687.95,57.35,155.00,0.00,0.00,0.00,57.35,0.00,0.00,10.89
6,G7,10.52,155.00,1687.95,57.35,155.00,0.00,0.00,0.00,57.35,0.00,0.00,10.89
7,G8,6.02,400.00,4356.00,1948.00,400.00,0.00,0.00,0.00,1948.00,0.00,0.00,10.89
8,G9,5.47,400.00,4356.00,2168.00,400.00,0.00,0.00,0.00,2168.00,0.00,0.00,10.89
9,G10,0.00,300.00,3267.00,3267.00,0.00,-300.00,-6831.00,-6831.00,-3564.00,-6831.00,-209.09,-inf


In the case that there is a need for upward regulation, 3 additional generators are called to generate, setting a higher price in the balancing market. Windfarms that exceeded their forecasted production are paid at a rate even higher than the day-ahead price, making it a huge win. Those wind generators that fell short on their forecasted production have to pay for the shortfall at a price higher than the day-ahead price. The generator with the forced outage has to pay for all the shortfall at a very high price, making it a huge loss.

When analysing the weighted price per MWh that each generators receives, it is clear that those who support the imbalance earn more than the day-ahead price per MWh, while those who contribute to the imbalance receive a lower price per MWh than the day-ahead price.

In [23]:
gen_s2.round(2)

,Unit,Cost_$_per_MWh,Scheduled_Generation_MW,DAM_Payments_$,DAM_Profit_$,Actual_Generation_MW,Deviation_MW,BM_Payments_$,BM_Profit_$,Total_Profit_$,Profit_Change_$,Profit_Change_%,Price_per_MWh
0,G1,13.32,-0.00,-0.00,0.00,0.00,0.00,0.00,-0.00,0.00,0.00,NaN,NaN
1,G2,13.32,-0.00,-0.00,0.00,0.00,0.00,0.00,-0.00,0.00,0.00,NaN,NaN
2,G3,20.70,-0.00,-0.00,0.00,0.00,0.00,0.00,-0.00,0.00,0.00,NaN,NaN
3,G4,20.93,-0.00,-0.00,0.00,0.00,0.00,0.00,-0.00,0.00,0.00,NaN,NaN
4,G5,26.11,-0.00,-0.00,0.00,0.00,0.00,0.00,-0.00,0.00,0.00,NaN,NaN
5,G6,10.52,155.00,1687.95,57.35,155.00,0.00,0.00,-0.00,57.35,0.00,0.0,10.89
6,G7,10.52,155.00,1687.95,57.35,155.00,0.00,0.00,-0.00,57.35,0.00,0.0,10.89
7,G8,6.02,400.00,4356.00,1948.00,400.00,0.00,0.00,0.00,1948.00,0.00,0.0,10.89
8,G9,5.47,400.00,4356.00,2168.00,400.00,0.00,0.00,0.00,2168.00,0.00,0.0,10.89
9,G10,0.00,300.00,3267.00,3267.00,300.00,0.00,0.00,0.00,3267.00,0.00,0.0,10.89


The contrary happens in the case of downward regulation, where the price in the balancing market is lower than the day-ahead price. In this case, those who support the imbalance pay to reduce their generation, but at a price lower than the day ahead price, while those that contribute to the imbalance are paid for their excess generation, but at a price lower than the day-ahead price.

Although the wind farms that produce less than they expected have to pay for their shortfall, they have to pay back at a price lower than the day-ahead price, meaning that what the price that they received for their total generation is higher than the day-ahead price. For the wind farms that produce more than expected, they are paid for their excess generation at a price lower than the day-ahead price, meaning that the price they receive for their total generation is lower than the day-ahead price.

In [24]:
print("Scenario 1 Summary: Upregulation")
print(gen_s1.sum()[['Scheduled_Generation_MW', 'Actual_Generation_MW', 'DAM_Payments_$', 'BM_Payments_$', 'DAM_Profit_$', 'BM_Profit_$', 'Total_Profit_$']].round(2))
print("\nScenario 2 Summary: Downregulation")
print(gen_s2.sum()[['Scheduled_Generation_MW', 'Actual_Generation_MW', 'DAM_Payments_$', 'BM_Payments_$', 'DAM_Profit_$', 'BM_Profit_$', 'Total_Profit_$']].round(2))

Scenario 1 Summary: Upregulation
Scheduled_Generation_MW     2517.97
Actual_Generation_MW        2517.97
DAM_Payments_$             27420.75
BM_Payments_$                   0.0
DAM_Profit_$               12562.99
BM_Profit_$                -4193.15
Total_Profit_$              8369.84
dtype: object

Scenario 2 Summary: Downregulation
Scheduled_Generation_MW     2517.97
Actual_Generation_MW        2517.97
DAM_Payments_$             27420.75
BM_Payments_$                   0.0
DAM_Profit_$               12562.99
BM_Profit_$                   82.22
Total_Profit_$             12645.21
dtype: object


When inspecting the summary of each scenario, it is clear that payments in the balancing market are zero-sum, but in the case of upward regulation the total BM profit is negative as a cheaper generator is having to pay a big price to cover its shortfall and pay for the more expensive generators, while in the case of downward regulation the total BM profit is positive as expensive generators are being paid to reduce their generation to allow cheaper generators to generate more.

#### Profits for generators under two pricing scheme

In this analysis we will change the pricing scheme for the balancing market. If generators are contributing to balancing they will be paid (or have to pay in case of downward regulation) at the day ahead price, otherwise they pay or are paid at the balancing market price.

In [25]:
def two_price_profit(base_df, p_gen_bm, p_wf_bm, imbalance_price, bm_model, dap):
    gen_df = base_df.copy()
    actual_gen = np.concatenate([
        p_gen_bm['solution'].values + bm_model.variables['upregulation'].solution.values - bm_model.variables['downregulation'].solution.values,
        p_wf_bm['solution'].values
    ])
    gen_df['Actual_Generation_MW'] = actual_gen
    gen_df['Deviation_MW'] = gen_df['Actual_Generation_MW'] - gen_df['Scheduled_Generation_MW']
    gen_df['Balancing_price'] = np.where(gen_df['Unit'].str.startswith('W') & ((gen_df['Deviation_MW'] * (dap - imbalance_price)) < 0), dap, imbalance_price)
    gen_df['BM_Payments_$'] = gen_df['Deviation_MW'] * gen_df['Balancing_price']
    gen_df['BM_Profit_$'] = gen_df['Deviation_MW'] * (gen_df['Balancing_price'] - gen_df['Cost_$_per_MWh'])
    gen_df['Total_Profit_$'] = gen_df['DAM_Profit_$'] + gen_df['BM_Profit_$']
    gen_df['Profit_Change_$'] = gen_df['Total_Profit_$'] - gen_df['DAM_Profit_$']
    gen_df['Profit_Change_%'] = (gen_df['Profit_Change_$'] / gen_df['DAM_Profit_$'].replace(0, np.nan)) * 100
    gen_df['Price_per_MWh'] = (gen_df['DAM_Payments_$'] + gen_df['BM_Payments_$'])/gen_df['Actual_Generation_MW']
    return gen_df

gen_s1_two_price = two_price_profit(base_df, p_gen_bm, p_wf_bm, s1_imbalance_price, bm_up, dap)
gen_s2_two_price = two_price_profit(base_df, p_gen_bm_down, p_wf_bm_down, s2_imbalance_price, bm_down, dap)

In [26]:
gen_s1_two_price.round(2)

,Unit,Cost_$_per_MWh,Scheduled_Generation_MW,DAM_Payments_$,DAM_Profit_$,Actual_Generation_MW,Deviation_MW,Balancing_price,BM_Payments_$,BM_Profit_$,Total_Profit_$,Profit_Change_$,Profit_Change_%,Price_per_MWh
0,G1,13.32,-0.00,-0.00,0.00,152.00,152.00,22.77,3461.04,1436.40,1436.40,1436.40,NaN,22.77
1,G2,13.32,-0.00,-0.00,0.00,152.00,152.00,22.77,3461.04,1436.40,1436.40,1436.40,NaN,22.77
2,G3,20.70,-0.00,-0.00,0.00,3.46,3.46,22.77,78.90,7.17,7.17,7.17,NaN,22.77
3,G4,20.93,-0.00,-0.00,0.00,0.00,0.00,22.77,0.00,0.00,0.00,0.00,NaN,NaN
4,G5,26.11,-0.00,-0.00,0.00,0.00,0.00,22.77,0.00,-0.00,0.00,0.00,NaN,NaN
5,G6,10.52,155.00,1687.95,57.35,155.00,0.00,22.77,0.00,0.00,57.35,0.00,0.00,10.89
6,G7,10.52,155.00,1687.95,57.35,155.00,0.00,22.77,0.00,0.00,57.35,0.00,0.00,10.89
7,G8,6.02,400.00,4356.00,1948.00,400.00,0.00,22.77,0.00,0.00,1948.00,0.00,0.00,10.89
8,G9,5.47,400.00,4356.00,2168.00,400.00,0.00,22.77,0.00,0.00,2168.00,0.00,0.00,10.89
9,G10,0.00,300.00,3267.00,3267.00,0.00,-300.00,22.77,-6831.00,-6831.00,-3564.00,-6831.00,-209.09,-inf


In [27]:
gen_s2_two_price.round(2)

,Unit,Cost_$_per_MWh,Scheduled_Generation_MW,DAM_Payments_$,DAM_Profit_$,Actual_Generation_MW,Deviation_MW,Balancing_price,BM_Payments_$,BM_Profit_$,Total_Profit_$,Profit_Change_$,Profit_Change_%,Price_per_MWh
0,G1,13.32,-0.00,-0.00,0.00,0.00,0.00,9.80,0.00,-0.00,0.00,0.00,NaN,NaN
1,G2,13.32,-0.00,-0.00,0.00,0.00,0.00,9.80,0.00,-0.00,0.00,0.00,NaN,NaN
2,G3,20.70,-0.00,-0.00,0.00,0.00,0.00,9.80,0.00,-0.00,0.00,0.00,NaN,NaN
3,G4,20.93,-0.00,-0.00,0.00,0.00,0.00,9.80,0.00,-0.00,0.00,0.00,NaN,NaN
4,G5,26.11,-0.00,-0.00,0.00,0.00,0.00,9.80,0.00,-0.00,0.00,0.00,NaN,NaN
5,G6,10.52,155.00,1687.95,57.35,155.00,0.00,9.80,0.00,-0.00,57.35,0.00,0.0,10.89
6,G7,10.52,155.00,1687.95,57.35,155.00,0.00,9.80,0.00,-0.00,57.35,0.00,0.0,10.89
7,G8,6.02,400.00,4356.00,1948.00,400.00,0.00,9.80,0.00,0.00,1948.00,0.00,0.0,10.89
8,G9,5.47,400.00,4356.00,2168.00,400.00,0.00,9.80,0.00,0.00,2168.00,0.00,0.0,10.89
9,G10,0.00,300.00,3267.00,3267.00,300.00,0.00,9.80,0.00,0.00,3267.00,0.00,0.0,10.89


In [28]:
print("Scenario 1 Summary: Upregulation")
print(gen_s1_two_price.sum()[['Scheduled_Generation_MW', 'Actual_Generation_MW', 'DAM_Payments_$', 'BM_Payments_$', 'DAM_Profit_$', 'BM_Profit_$', 'Total_Profit_$']].round(2))
print("\nScenario 2 Summary: Downregulation")
print(gen_s2_two_price.sum()[['Scheduled_Generation_MW', 'Actual_Generation_MW', 'DAM_Payments_$', 'BM_Payments_$', 'DAM_Profit_$', 'BM_Profit_$', 'Total_Profit_$']].round(2))

Scenario 1 Summary: Upregulation
Scheduled_Generation_MW     2517.97
Actual_Generation_MW        2517.97
DAM_Payments_$             27420.75
BM_Payments_$               -257.08
DAM_Profit_$               12562.99
BM_Profit_$                -4450.23
Total_Profit_$              8112.76
dtype: object

Scenario 2 Summary: Downregulation
Scheduled_Generation_MW     2517.97
Actual_Generation_MW        2517.97
DAM_Payments_$             27420.75
BM_Payments_$                -38.91
DAM_Profit_$               12562.99
BM_Profit_$                   43.31
Total_Profit_$              12606.3
dtype: object


Under the two-price scheme, wind farms are not benefited for their contribution to reducing the imbalance, which intervines the market and now the Balancing Payments are not zero-sum. In both cases there is an additional BM payment that would be paid to the TSO. This is meant to descourage wind generators to deviate from their forecasted production.

Personally, I don't see the point of this two-price scheme, wind generators would need very accurate forecasts of imbalance to be able to make these decisions. Furthermore they rely on forecasts and the variable nature of their resource, which means that a good model will sometimes be long and sometimes short, but on average they will be close to their forecasted production.